# Basics for playing with Hidden Markov Model

This notebook is designed for HKU's BIOC3605 (Sequence Bioinformatics), and can be opened on Google CoLab via this [link](https://colab.research.google.com/github/StatBiomed/bioinfo-book/blob/main/examples/HMM_basics_BIOC3605.ipynb).


In [1]:
import numpy as np
import matplotlib.pyplot as plt

## Toy example 1: weather and clothing

In [2]:
psi = np.array([0.6, 0.3, 0.1])
A = np.array([
    [0.6, 0.3, 0.1],
    [0.4, 0.3, 0.3],
    [0.1, 0.4, 0.5]
])
baselik = np.array([
    [0.61, 0.38, 0.01],
    [0.24, 0.51, 0.25],
    [0.01, 0.11, 0.88]
])

#### Generate a random path of states

In [3]:
def path_generator(prior, transitions, path_length, state_keys=None):
    path_random = []
    if state_keys is None:
        state_keys = np.range(len(prior))

    pre_state = None
    for i in range(path_length):
        # initial state
        if i == 0:
            _state = np.random.choice(len(prior), p = prior)
            path_random.append(state_keys[_state])
        else:

            _state = np.random.choice(len(prior), p = transitions[pre_state, :])
            path_random.append(state_keys[_state])
        pre_state = _state + 0

    return path_random

In [4]:
a_random_path = path_generator(psi, A, 5, state_keys = ["Hot", "Mild", "Cold"])
print(a_random_path)

['Hot', 'Mild', 'Cold', 'Cold', 'Mild']


In [5]:
for i in range(4):
    a_random_path = path_generator(psi, A, 7, ["Hot", "Mild", "Cold"])
    print(a_random_path)

['Cold', 'Mild', 'Cold', 'Mild', 'Hot', 'Cold', 'Mild']
['Hot', 'Hot', 'Mild', 'Mild', 'Cold', 'Cold', 'Cold']
['Hot', 'Hot', 'Hot', 'Hot', 'Hot', 'Hot', 'Hot']
['Cold', 'Mild', 'Mild', 'Hot', 'Cold', 'Mild', 'Mild']


#### Evaluate the probability of a certain state path

* Observed values in a sequence: [“Semi Casual”, “Semi Casual”, “winter apparel”, “Casual”, “Semi Casual”]

* example state path: [“Hot”, “Hot”, “Cold”, “Hot”, “Hot”]

In [6]:
toy1_states = [0, 0, 2, 0, 0]
toy1_values = [1, 1, 2, 0, 1]

In [7]:
def score_HMM_path(prior, transitions, emissions, seq_values, seq_states):
    priors_list = []
    likeli_list = []
    pre_state = None
    for i in range(len(seq_values)):
        _value = seq_values[i]
        _state = seq_states[i]
        if pre_state is None:
            _prior = prior[_state]
        else:
            _prior = transitions[pre_state, _state]

        priors_list.append(_prior)
        likeli_list.append(emissions[_state, _value])
        pre_state = _state + 0

    return np.array(priors_list), np.array(likeli_list)

In [8]:
toy1_priors_list, toy1_likeli_list = score_HMM_path(
    psi, A, baselik, toy1_values, toy1_states
)

toy1_priors_list, toy1_likeli_list

(array([0.6, 0.6, 0.1, 0.1, 0.6]), array([0.38, 0.38, 0.88, 0.61, 0.38]))

In [9]:
np.prod(toy1_priors_list * toy1_likeli_list)

np.float64(6.362342553599999e-05)

## Toy example 2: 5' splice site

In [10]:
psi = np.array([1/3, 1/3, 1/3])

A = np.array([
    [0.9, 0.1, 0],
    [0, 0, 1],
    [0, 0, 0.9]
])

baselik = np.array([
    [1/4, 1/4, 1/4, 1/4],
    [0.05, 0, 0.95, 0],
    [0.4, 0.1, 0.1, 0.4]
])

print(psi)
print(A)
print(baselik)

[0.33333333 0.33333333 0.33333333]
[[0.9 0.1 0. ]
 [0.  0.  1. ]
 [0.  0.  0.9]]
[[0.25 0.25 0.25 0.25]
 [0.05 0.   0.95 0.  ]
 [0.4  0.1  0.1  0.4 ]]


In [11]:
A[0, 1]

np.float64(0.1)

In [12]:
seqSS = "CTTCATGTGAAAGCAGACGTAAGTCA"
path0 = "EEEEEEEEEEEEEEEEEE5IIIIIII"
path1 = "EEEEEE5IIIIIIIIIIIIIIIIIII"
path2 = "EEEEEEEE5IIIIIIIIIIIIIIIII"

In [13]:
toy2_values = ['ACGT'.index(x) for x in seqSS]
toy2_states = [ 'E5I'.index(x) for x in path0]

toy2_priors_list, toy2_likeli_list = score_HMM_path(
    psi, A, baselik, toy2_values, toy2_states
)

toy2_priors_list, toy2_likeli_list

(array([0.33333333, 0.9       , 0.9       , 0.9       , 0.9       ,
        0.9       , 0.9       , 0.9       , 0.9       , 0.9       ,
        0.9       , 0.9       , 0.9       , 0.9       , 0.9       ,
        0.9       , 0.9       , 0.9       , 0.1       , 1.        ,
        0.9       , 0.9       , 0.9       , 0.9       , 0.9       ,
        0.9       ]),
 array([0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25,
        0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.95, 0.4 , 0.4 , 0.4 ,
        0.1 , 0.4 , 0.1 , 0.4 ]))

In [14]:
np.sum(np.log(toy2_priors_list) + np.log(toy2_likeli_list))

np.float64(-40.015704881696614)

### Misc

In [15]:
def onehot_encode(sequence, keys='CAGT'):
    mapping = dict(zip(keys, range(len(keys))))
    int_encoded = [mapping[i] for i in sequence]
    return np.eye(len(keys))[int_encoded]

seq = "CTTCATGTGAAAGCAGACGTAAGTCA"
seq_onehot = onehot_encode(seq)
print(seq_onehot)

[[1. 0. 0. 0.]
 [0. 0. 0. 1.]
 [0. 0. 0. 1.]
 [1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 0. 1.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]
 [0. 0. 1. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 1. 0. 0.]
 [1. 0. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]
 [1. 0. 0. 0.]
 [0. 1. 0. 0.]]


psi = np.array([0, 1/3, 1/3, 1/3, 0])
A = np.array([
    [1, 0, 0, 0, 0],
    [0, 0.9, 0.1, 0, 0],
    [0, 0, 0, 1, 0],
    [0, 0, 0, 0.9, 0.1],
    [0, 0, 0, 0, 1]
])
baselik = np.array([
    [1/4, 1/4, 1/4, 1/4],
    [1/4, 1/4, 1/4, 1/4],
    [0.05, 0, 0.95, 0],
    [0.4, 0.1, 0.1, 0.4],
    [1/4, 1/4, 1/4, 1/4]
])

print(psi)
print(A)
print(baselik)